# AB Aur, reduced step by step

`nirc2pol-reduce night.toml` runs this whole thing in one call. This notebook
takes it apart, so you can see each stage, look at what it produced, and
change one without rewriting the rest — which is what you want when a
reduction needs to depart from the recipe.

The stages, and the order, are the ones in `nirc2pol.polmode.run`. If you
change the order here you are no longer reducing the same way the pipeline
does, and the two will disagree for reasons that are hard to find later.

**The paths below point at a real night on `mueller`.** Change
`raw_data_folder` and `reductions_root` to your own before running.

In [ ]:
import glob, logging, os

import matplotlib.pyplot as plt
import numpy as np

from nirc2pol.instruments import nirc2
from nirc2pol.instruments.nirc2 import NIRC2PolarimetryData
from nirc2pol.polarimetry import (ProductWriter, aperture_polarization,
                                  build_stokes_cubes, fit_ip_uphi,
                                  fit_ip_uphi_all, mean_ip,
                                  median_stokes_cube, radial_stokes)
from nirc2pol.reduction import (make_master_darks, make_master_flats,
                                make_master_masks, make_master_skies,
                                reduce_frame)
from nirc2pol.reduction.config import ReductionConfig
from nirc2pol.utils import (ObslogPaths, load_frames, load_rejects,
                            save_frames, select_frames, start_reduction_log)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("ab_aur_manual")

## 0. The config

Every choice in one object, so the run can be described, written down and
repeated. `nirc2pol-reduce --template > night.toml` prints the same fields
with their documentation.

In [ ]:
cfg = ReductionConfig(
    # where the frames are, and where this reduction goes -- two
    # different things, so two keys
    raw_data_folder="/home/shared/exoserver/NIRC2_Pol/20251207",
    reductions_root="/home/blewis/reductions/ab_aur_Lp_20251207",
    date="2025-12-07",
    target="AB_Aur",

    # everything read off disk: the science runs, plus the darks and sky
    # flats that sit between them
    raw_range=[857, 993],
    # ...and the science frames that become products: three runs
    select_frame_range=[[857, 900], [915, 930], [932, 963]],

    # This night was not dithered, so the background comes off a box in a
    # source-free corner. A dithered L' night wants ["dither", "annulus"]:
    # the dither removes the thermal pedestal and its structure, the annulus
    # removes the residual it leaves.
    background_method="mean_box",
    background_box=[25, 350, 50, 400],

    register_method="min",              # AB Aur's L' core is saturated

    fast_axis_method="fixed",           # pinned, not fitted here
    theta_off=-13.1,                    # the butterfly value

    ip_method=None,                     # left uncorrected
    ip_mask_radius=22.0,

    save_preproc=True,                  # keep masters + corrected frames
    save_individual_cycles=False,
)

# A config that lives only in memory cannot be pointed at, so write it out.
# With no argument it lands in reductions_root as reduction_<date>.toml --
# beside the products, and the file you would hand to nirc2pol-reduce to
# repeat this run.
config_path = cfg.to_toml()
print("config written to", config_path)

## 1. Instrument, paths, rejects, log

`configure` copies the config's choices onto the instrument and hands it
back. Note what it does *not* do: it leaves the beam cutout alone unless you
named one, because the instrument already carries a nominal value per band
and overwriting that with `None` would leave nothing to cut with.

In [ ]:
instrument = cfg.configure(NIRC2PolarimetryData())

paths = ObslogPaths(cfg.reductions_root, cfg.date)
paths.make_folders()
rejects = load_rejects(paths.rejects_file)
print(f"{len(rejects)} rejected frame(s) from {paths.rejects_file}")

# Everything the reduction reports lands in one file beside the products --
# which flat it matched, which sky, any fallback -- so the choices survive
# the notebook session.
run_log = start_reduction_log(paths.log_file)
run_log.settings(instrument=type(instrument).__name__,
                 background=instrument.describe_background(),
                 config=config_path, **cfg.describe())
print("log:", run_log.path)

## 2. Sort the raw frames

The frames are symlinked into `reductions_root/raw` rather than read in
place, so a reduction records exactly which files it saw without copying
them or writing to the archive.

In [ ]:
raw_files = paths.link_raw_frames(cfg.raw_data_folder,
                                  frame_range=cfg.raw_range)
print(f"{len(raw_files)} raw files linked into {paths.raw_folder}")

sorted_files = instrument.sort_frames(raw_files)
for kind, files in sorted_files.items():
    print(f"   {kind:12s} {len(files)}")

## 3. Master darks, flats and skies

Flats are matched on `FILTER` before anything else, and the log records which
one calibrated each frame — worth reading, because a flat from the wrong band
is a plausible-looking result rather than an error.

In [ ]:
darks = load_frames(sorted_files["darks"], rejects=rejects)
master_darks, dark_masks = make_master_darks(darks, instrument=instrument)
if master_darks and cfg.save_preproc:
    save_frames(paths.darks_file, master_darks)

master_flats, flat_masks = make_master_flats(
    load_frames(sorted_files["flats_dome"], rejects=rejects),
    load_frames(sorted_files["flats_sky"], rejects=rejects),
    master_darks, instrument=instrument)
if master_flats and cfg.save_preproc:
    save_frames(paths.flats_file, master_flats)

master_skies = None
if cfg.use_master_skies:
    master_skies, _ = make_master_skies(
        load_frames(sorted_files["flats_sky"], rejects=rejects),
        master_darks, instrument=instrument)
    if master_skies and cfg.save_preproc:
        save_frames(paths.skies_file, master_skies)

master_masks = make_master_masks(dark_masks, flat_masks)

print(f"{len(master_darks)} master darks, {len(master_flats)} master flats")
for f in master_flats:
    print(f"   {f['FILTER']:16s} FLATTYPE {f['FLATTYPE']:5s} "
          f"polarimetric {f.get('POLFLAT')}  from {f['NFRAMES']} frames")

## 4. Which frames this reduction covers

Selection comes **before** the reduce loop, not after. `sort_frames`
classifies by elimination, so acquisition and engineering frames in other
bands end up in the science bucket; reducing those first wastes the work and
fails on the flat match.

In [ ]:
bad_pixel_mask = instrument.bad_pixel_mask()   # loads a FITS; read once

sci_frames = load_frames(sorted_files["sci"], rejects=rejects)
sci_frames = select_frames(sci_frames, target=cfg.select_target,
                           frame_range=cfg.select_frame_range)
paths.check_frame_dates(sci_frames)            # folder date vs DATE-OBS
nirc2.make_frametable(sci_frames, paths.table_file)
print(f"{len(sci_frames)} frames selected")

## 5. Reduce them

Dark, flat, bad pixels — per frame, and nothing polarimetric yet.

In [ ]:
reduced_frames = []
for frame in sci_frames:
    reduced = reduce_frame(
        frame, master_flats, master_darks, master_skies, master_masks,
        bad_pixel_mask=bad_pixel_mask,
        required_flat_types=instrument.required_flat_types,
        default_required_flat_type=instrument.default_required_flat_type,
        gain=instrument.gain(frame),
        saturation_limit=instrument.saturation_limit(frame),
    )
    if cfg.save_preproc:
        reduced.save(os.path.join(paths.reduced_folder, reduced["RED-FN"]))
    reduced_frames.append(reduced)

print(f"{len(reduced_frames)} frames reduced")

## 6. The beam cutout, then HWP cycles

There is no geometry to fit. `split_beams` slices two integers out of the
detector, and those come from the `[beam_geometry]` table in `nirc2.toml` —
**approximate on purpose**. The two Wollaston beams are rotated relative to
each other by about 0.37°, so their separation depends on where in the field
the source sits (2.2 px across a 300 px dither throw), and no single pair of
integers is right at more than one position.

Whatever the cut leaves between the beams is measured and removed **per
frame** by `align_beams`, inside registration. That is both simpler and
strictly more accurate than choosing the pair well.

In [ ]:
print("beam cutout:", instrument.top_row_start, instrument.beam_x_offset,
      "(nominal; align_beams removes the residual per frame)")

cycles = instrument.match_modulator_cycles(reduced_frames)
print(f"{len(cycles)} complete HWP cycles")
print("cycle 0 HWP angles:",
      [round(instrument.modulator_angle(f), 1) for f in cycles[0]])

## 7. Fast axis offset and instrumental polarization

Two separate choices. `theta_off` is pinned here to the butterfly value;
`examples/polarized_standard.ipynb` measures it the other way, against a
catalogue angle.

The leakage is left uncorrected on this night. It is of order 1–2% on NIRC2,
so anything you read off the products below carries it.

In [ ]:
theta_off = cfg.theta_off
ip = None
dd_kwargs = {"register_method": cfg.register_method}

if cfg.ip_method == "fit_uphi_all":
    ip = fit_ip_uphi_all(instrument, cycles, theta_off,
                         mask_radius=cfg.ip_mask_radius, **dd_kwargs)
    print("IP:", ip.describe())
elif cfg.ip_method == "fit_uphi_per_cycle":
    ip = [fit_ip_uphi(instrument, c, theta_off,
                      mask_radius=cfg.ip_mask_radius, **dd_kwargs)
          for c in cycles]
    print("IP (mean of per-cycle):", mean_ip(ip).describe())

instrument.fast_axis_offset = theta_off
print(f"theta_off = {theta_off} deg, ip_method = {cfg.ip_method}")

## 8. Stokes cubes

Beam splitting, alignment, background, registration and the double difference
all happen inside the builder, once per cycle.

`build_stokes_cubes` also applies any frame-level background the config
declares but the frames have not had — a dither, which runs on whole frames
before the beams are cut. This night uses `mean_box`, which is a per-beam
stage, so nothing extra happens here.

In [ ]:
stokes_cubes = build_stokes_cubes(instrument, cycles,
                                  fast_axis_offset=theta_off, ip=ip,
                                  crop_size=cfg.crop_size, **dd_kwargs)
median_cube = median_stokes_cube(stokes_cubes)
print(f"{len(stokes_cubes)} cycle cubes -> median {median_cube.shape}")

## 9. Products

`THETAOFF` is written into the cycle header by `double_difference`, so the
products inherit it without anyone stamping it on by hand.

In [ ]:
header = cycles[0][0].header.copy()

writer = ProductWriter(paths.sequences_folder, target=cfg.target)
if cfg.save_individual_cycles:
    writer.save_stokes_cycles(stokes_cubes, cycles, header=header)
writer.save_median_stokes(median_cube, header=header)
writer.save_derived_products(median_cube, header=header,
                             derived=cfg.save_derived_quantities,
                             radial=cfg.save_radial_stokes)

run_log.finish()
print(f"\nproducts in {writer.output_dir}")
print(f"log: {run_log.path} ({run_log.warnings} warnings)")

## 10. Look at it

`U_phi` is the null channel: for a tangentially polarized disk it should hold
nothing, so what is in it is the honest measure of what the reduction got
wrong.

In [ ]:
qphi, uphi = radial_stokes(median_cube[1], median_cube[2])

v = np.nanpercentile(np.abs(uphi[np.isfinite(uphi)]), 99.6)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a_, img, title in [(ax[0], qphi, "$Q_\\phi$"),
                       (ax[1], uphi, "$U_\\phi$ (null channel)")]:
    im = a_.imshow(img, origin="lower", vmin=-v, vmax=3 * v, cmap="inferno")
    a_.set_title(title, fontsize=12)
    fig.colorbar(im, ax=a_, fraction=0.046)
fig.suptitle(f"AB Aur {cfg.date}, theta_off = {theta_off} deg, "
             f"ip_method = {cfg.ip_method}", fontsize=13)
fig.tight_layout()

finite = np.isfinite(uphi) & np.isfinite(qphi)
print(f"Q_phi max {np.nanmax(qphi):.1f},  "
      f"U_phi std {np.nanstd(uphi[finite]):.3f}")

## 11. Measure the disk polarization

`aperture_polarization` integrates `q`, `u` over an aperture and reports
`p` and the position angle.

**Give it a background annulus.** Q and U are differences, so the sky is
expected to cancel and does not quite: the residual is small per pixel and
enormous once summed over a large aperture. And read the numbers as a series
— aperture losses cannot bias `p`, since `q`, `u` and `I` are integrated over
the same pixels, so a `p` that moves with radius is telling you about the
background rather than the disk.

In [ ]:
print(f"  {'r [px]':>7} {'p [%]':>9} {'theta [deg]':>12}")
for radius in (25, 50, 75, 100, 150):
    res = aperture_polarization(median_cube, radius=radius,
                                background=(180, 220))
    print(f"  {radius:7.0f} {100 * res['p']:9.3f} {res['theta']:12.2f}")